# 🟡 Chapter 4: Representing Data and Engineering Features
**Reference:** *Introduction to Machine Learning with Python: A Guide for Data Scientists* by Andreas C. Müller & Sarah Guido

---

## 4.1 The Importance of Feature Engineering
Up to this point, we have assumed that our data is presented as a nice, clean matrix of floating-point numbers where each column represents a continuous feature. However, in real-world scenarios, data rarely arrives in this format. The most common data type encountered in commercial applications is **Categorical Features** (also known as discrete features), which represent properties like types of jobs, product categories, or geographic locations.

How you represent your data is the single most critical factor in determining the real-world success of a Machine Learning model. The process of modifying existing features, mapping data types, or engineering entirely new structural components to aid linear or non-linear models is known as **Feature Engineering**. 

An optimal data representation chosen by an engineer can allow a conceptually simple linear algorithm to easily outperform a highly complex, computationally heavy deep learning architecture trained on raw, unengineered features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Ensure plots display cleanly inline
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
print("Feature Engineering toolkit successfully loaded.")

## 4.2 Categorical Variables: One-Hot-Encoding (Dummy Variables)
The absolute majority of machine learning algorithms in Scikit-learn (such as `LinearRegression` or `LogisticRegression`) are strictly mathematical equations that require input dimensions to be represented entirely as continuous numbers. They cannot interpret text strings or abstract object labels directly.

To handle categorical variables, we transform them using **One-Hot Encoding**, also known as **Dummy Variables**. This process strips away the single categorical text column and replaces it with multiple binary columns (containing only `0` or `1`), where each new column corresponds to one unique state within the original category. If an entity belongs to a specific category, that column is set to `1` ("hot"), while all other engineered dummy columns are set to `0` ("cold").

### Using Pandas `get_dummies` vs. Scikit-learn `OneHotEncoder`
The easiest way to perform one-hot encoding on tabular datasets is using the built-in `pd.get_dummies()` function from Pandas. However, for Production Pipelines where unseen test samples might arrive with missing or novel categories, Scikit-learn's `OneHotEncoder` class is preferred as it stores the structural vocabulary state during training.

In [ ]:
# Create an industrial-style synthetic dataset with mixed features
data = {
    'Name': ['Alex', 'Brian', 'Chloe', 'Daniel'],
    'Occupation': ['Student', 'Programmer', 'Student', 'Designer'],
    'Age': [21, 25, 20, 28]
}
df = pd.DataFrame(data)

print("Original DataFrame:")
display(df)

# Apply one-hot encoding using pandas get_dummies
# We explicitly track 'Occupation' to showcase categorical conversion
df_encoded = pd.get_dummies(df, columns=['Occupation'])

print("\nDataFrame after One-Hot Encoding:")
display(df_encoded)

print("\nAnalysis:")
print("Notice how the single 'Occupation' column has expanded into three continuous binary features:")
print("Occupation_Designer, Occupation_Programmer, and Occupation_Student.")

## 4.3 Interaction Terms and Polynomial Features
Another highly effective method for enriching data representation, particularly when utilizing linear models, is expanding features into **Polynomial Dimensions** or adding **Interaction Terms**.

Linear models are mathematically limited because they fit data by calculating a simple weighted sum of individual properties, assuming each feature behaves independently. For example, if a dataset contains feature $x_1$, a standard linear model can only draw a straight line to map relationships ($y = w_1x_1 + b$). If the real-world data curves or waves, simple linear regression will suffer severely from **Underfitting**.

To solve this, we mathematically expand our feature matrix by computing polynomial combinations up to a specified degree ($x_1^2, x_1^3, x_1^4, \dots$). Furthermore, we can compute **Interaction Features**, which multiply distinct columns together ($x_1 \times x_2$). This explicitly provides the model with a metric to look at the *combined effect* of variables, allowing a simple linear framework to capture non-linear relationships and fit highly curved datasets.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# 1. Generate non-linear data (A complex sine wave curve with Gaussian noise)
X = np.linspace(-3, 3, 100).reshape(-1, 1)
y = np.sin(X[:, 0]) + np.random.normal(0, 0.2, X.shape[0])

# 2. Train a baseline Standard Linear Regression model (Underfitting scenario)
reg = LinearRegression().fit(X, y)
y_pred_linear = reg.predict(X)

# 3. Engineer Polynomial Features up to Degree 5
# include_bias=False prevents adding a column of all 1s (which is redundant for OLS)
poly = PolynomialFeatures(degree=5, include_bias=False)
X_poly = poly.fit_transform(X)

# 4. Fit a new Linear Regression model utilizing the newly engineered polynomial spaces
reg_poly = LinearRegression().fit(X_poly, y)
y_pred_poly = reg_poly.predict(X_poly)

# 5. Visualization of the geometric transformations
plt.figure(figsize=(12, 6))
plt.plot(X, y, 'o', c='gray', markersize=5, alpha=0.7, label='Raw Observations (y)')
plt.plot(X, y_pred_linear, c='red', lw=3, linestyle='--', label='Linear Regression (Standard - Underfit)')
plt.plot(X, y_pred_poly, c='green', lw=3, label='Linear Regression (Engineered Polynomial Degree 5)')

plt.title("Overcoming Linear Underfitting with Engineered Polynomial Dimensions", fontsize=14)
plt.xlabel("Input Feature Matrix (X)", fontsize=12)
plt.ylabel("Target Vector (y)", fontsize=12)
plt.legend(loc='upper right', frameon=True)
plt.show()

## 4.4 Automatic Feature Selection
While expanding our design matrices via polynomial engineering or binning provides algorithms with immense flexibility, it also carries a significant risk: **The Curse of Dimensionality**. Adding dozens of higher-degree curves or cross-multiplied interaction columns can quickly introduce highly correlated noise or cause severe **Overfitting**.

To combat this, we implement **Automatic Feature Selection**. The textbook highlights three main strategies to isolate the most valuable signal:
1. **Univariate Statistics (SelectPercentile / SelectKBest):** Evaluates each feature independently to check if there is a statistically significant relationship with the target vector using an ANOVA (Analysis of Variance) F-test.
2. **Model-Based Selection:** Employs a supervised core algorithm (like a Random Forest or Linear Regularization) to compute explicit feature importances, discarding variables below a chosen threshold.
3. **Iterative Selection (RFE - Recursive Feature Elimination):** Trains a series of models, dropping the least significant variable step-by-step until the target feature count is met.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectPercentile, f_classif

# Load real-world Breast Cancer dataset (30 native dimensions)
cancer = load_breast_cancer()

# Simulate a high-dimensional noisy system:
# We generate 50 completely random columns of mathematical noise and append them to the data
rng = np.random.RandomState(42)
noise = rng.normal(size=(len(cancer.data), 50))
X_w_noise = np.hstack([cancer.data, noise])

# Apply Univariate Feature Selection (SelectPercentile) to keep only the top 50% best features
# Score function 'f_classif' evaluates classification variance across targets
select = SelectPercentile(score_func=f_classif, percentile=50)
select.fit(X_w_noise, cancer.target)
X_selected = select.transform(X_w_noise)

print(f"Shape of Design Matrix BEFORE Selection (Features + Noise): {X_w_noise.shape}")
print(f"Shape of Design Matrix AFTER Selection (Signal Isolated):  {X_selected.shape}\n")

# Generate a support mask visual matrix to observe which features were chosen
mask = select.get_support()

plt.figure(figsize=(16, 2))
plt.matshow(mask.reshape(1, -1), cmap='gray_r', fignum=1)
plt.xlabel("Engineered Feature Array Index Space", fontsize=12)
plt.yticks(())
plt.title("Univariate Support Mask Map (Black = Retained, White = Discarded)\n", fontsize=14)
plt.show()

print("Statistical Conclusion:")
print("The algorithm successfully kept almost all the native, high-signal features (indices 0 to 29),")
print("while systematically discarding the vast majority of the randomly injected noise columns (indices 30 to 79).")

## 4.5 Summary of Essential Takeaways
According to the text, matching data representation to the specific mathematical assumptions of your chosen algorithm is paramount:
- **One-Hot Encoding** is non-negotiable for enabling linear models to parse categorical attributes cleanly without implying false ordinal relationships.
- **Polynomial Features** allow linear frameworks to bend their decision boundaries, mapping complex curvatures without forcing you to switch to black-box non-linear architectures.
- **Feature Selection** acts as a critical algorithmic filter, stripping out dimensions that do not contribute variance to the target variable and protecting systems against overfitting.